# RFQ & Trade Execution Analysis

This notebook demonstrates cross-dataset analysis by joining RFQ (Request for Quote) events with trade execution records. This is a practical workflow for OTC desk analysts evaluating counterparty behavior and execution quality.

**What you'll learn:**
- Loading and exploring multiple datasets
- Cross-dataset joins (RFQ events + trade records)
- Counterparty acceptance rate analysis
- Execution slippage measurement (quoted vs executed price)
- RFQ timing analysis with DuckDB SQL
- Working with computed RFQ analytics features

**Datasets:** `rfq_events` (60 rows) + `trade_records` (14 rows) + `rfq_analytics` + `trade_factors`

## 1. Pipeline Setup

In [ ]:
import subprocess
import os
from pathlib import Path

# Navigate to the project root (parent of notebooks/)
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
elif not Path("config/datasets.yaml").exists():
    root = Path.cwd()
    while not (root / "config" / "datasets.yaml").exists() and root != root.parent:
        root = root.parent
    os.chdir(root)

subprocess.run(["adp", "init"], capture_output=True, check=True)

# Ingest and snapshot both datasets (RFQ lifecycle events + trade executions)
for dataset in ["rfq_events", "trade_records"]:
    subprocess.run(["adp", "ingest", dataset, "--force"], capture_output=True, check=True)
    subprocess.run(["adp", "snapshot", "create", dataset], capture_output=True, check=True)

# Build feature sets: rfq_analytics (quantity trends) and trade_factors (execution metrics)
subprocess.run(["adp", "features", "build", "rfq_events", "rfq_analytics"], capture_output=True, check=True)
subprocess.run(["adp", "features", "build", "trade_records", "trade_factors"], capture_output=True, check=True)
print("Pipeline complete: rfq_events + trade_records with features")

In [ ]:
import polars as pl
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from adp import (
    load_dataset,
    load_features,
    query_dataset,
    list_datasets,
)

pl.Config.set_tbl_rows(25)
pl.Config.set_fmt_str_lengths(50)

## 2. RFQ Lifecycle Overview

Each RFQ goes through a lifecycle: **requested** -> **quoted** -> **accepted/rejected**. Let's explore the data structure and summarize the flow.

In [ ]:
rfq_df = load_dataset("rfq_events").collect()

print(f"Shape: {rfq_df.shape}")
print(f"Columns: {rfq_df.columns}")
print(f"\nUnique RFQs: {rfq_df['rfq_id'].n_unique()}")
print(f"Instruments: {rfq_df['instrument'].unique().to_list()}")
print(f"Counterparties: {rfq_df['counterparty'].unique().to_list()}")

print("\n=== Status Distribution ===")
print(
    rfq_df.group_by("status")
    .agg(pl.col("event_id").count().alias("count"))
    .sort("status")
)

print("\nSample RFQ lifecycle (RFQ_001):")
print(
    rfq_df.filter(pl.col("rfq_id") == "RFQ_001")
    .select(["event_id", "rfq_id", "status", "timestamp", "quoted_price"])
)

## 3. Counterparty Acceptance Analysis

Which counterparties have the best acceptance rates? How do they compare in terms of volume?

In [ ]:
# Filter to final-state events only (accepted/rejected)
outcomes = rfq_df.filter(pl.col("status").is_in(["accepted", "rejected"]))

# Use n_unique consistently to count distinct RFQs (not rows)
acceptance = (
    outcomes
    .group_by("counterparty")
    .agg([
        pl.col("rfq_id").n_unique().alias("total_rfqs"),
        pl.col("rfq_id").filter(pl.col("status") == "accepted").n_unique().alias("accepted"),
        pl.col("rfq_id").filter(pl.col("status") == "rejected").n_unique().alias("rejected"),
        pl.col("requested_qty").mean().round(4).alias("avg_qty"),
    ])
    .with_columns(
        (pl.col("accepted") / pl.col("total_rfqs") * 100)
        .round(1)
        .alias("acceptance_rate_pct")
    )
    .sort("acceptance_rate_pct", descending=True)
)

print("=== Acceptance Rate by Counterparty ===")
print(acceptance)

In [ ]:
# Bar chart: acceptance rate by counterparty
counterparties = acceptance["counterparty"].to_list()
rates = acceptance["acceptance_rate_pct"].to_list()
colors = [plt.cm.Set2(i) for i in range(len(counterparties))]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(counterparties, rates, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Counterparty")
ax.set_ylabel("Acceptance Rate (%)")
ax.set_title("RFQ Acceptance Rate by Counterparty")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 4. RFQ Timing Analysis with DuckDB

Use SQL to compute the response time (requested -> quoted) and decision time (quoted -> accepted/rejected) for each RFQ.

In [ ]:
timing = query_dataset(
    "rfq_events",
    """
    WITH rfq_lifecycle AS (
        SELECT
            rfq_id,
            instrument,
            counterparty,
            MIN(CASE WHEN status = 'requested' THEN timestamp END) AS requested_at,
            MIN(CASE WHEN status = 'quoted' THEN timestamp END) AS quoted_at,
            MIN(CASE WHEN status IN ('accepted', 'rejected') THEN timestamp END) AS decided_at,
            MAX(CASE WHEN status IN ('accepted', 'rejected') THEN status END) AS outcome
        FROM dataset
        GROUP BY rfq_id, instrument, counterparty
    )
    SELECT
        counterparty,
        COUNT(*) AS rfq_count,
        SUM(CASE WHEN outcome = 'accepted' THEN 1 ELSE 0 END) AS accepted,
        ROUND(AVG(EPOCH(quoted_at - requested_at)), 1) AS avg_quote_time_sec,
        ROUND(AVG(EPOCH(decided_at - quoted_at)), 1) AS avg_decision_time_sec,
        ROUND(AVG(EPOCH(decided_at - requested_at)), 1) AS avg_total_time_sec
    FROM rfq_lifecycle
    GROUP BY counterparty
    ORDER BY rfq_count DESC
    """,
)

print("=== RFQ Lifecycle Timing by Counterparty ===")
print(timing)

## 5. Trade Execution Quality

Join accepted RFQs with their trade executions to measure slippage — the difference between the quoted price and the actual execution price, expressed in basis points.

**Sign convention:** Positive slippage = adverse execution (buyer paid more or seller received less than quoted).

In [ ]:
trade_df = load_dataset("trade_records").collect()
print(f"Trade records: {trade_df.shape}")

# Get accepted RFQs with their quoted prices (only the field not already in trade_df)
accepted_rfqs = (
    rfq_df
    .filter(pl.col("status") == "accepted")
    .select(["rfq_id", "quoted_price"])
)

# Join trades with their RFQ quotes to compare execution vs quoted price.
# Slippage sign convention: positive = adverse to the trader.
# For buys, paying more than quoted is adverse; for sells, receiving less is adverse.
slippage = (
    trade_df
    .join(accepted_rfqs, on="rfq_id")
    .with_columns(
        pl.when(pl.col("side") == "buy")
        .then((pl.col("price") - pl.col("quoted_price")) / pl.col("quoted_price") * 10000)
        .otherwise((pl.col("quoted_price") - pl.col("price")) / pl.col("quoted_price") * 10000)
        .round(2)
        .alias("slippage_bps")
    )
)

print("\n=== Trade Execution Slippage ===")
print(
    slippage.select(
        ["trade_id", "rfq_id", "instrument", "side",
         "quoted_price", "price", "slippage_bps"]
    )
)

In [ ]:
# Slippage summary by side
print("=== Average Slippage by Side ===")
print(
    slippage
    .group_by("side")
    .agg([
        pl.col("slippage_bps").mean().round(2).alias("avg_slippage_bps"),
        pl.col("slippage_bps").std().round(2).alias("std_slippage_bps"),
        pl.col("slippage_bps").count().alias("trade_count"),
    ])
)

# Slippage summary by counterparty
print("\n=== Average Slippage by Counterparty ===")
print(
    slippage
    .group_by("counterparty")
    .agg([
        pl.col("slippage_bps").mean().round(2).alias("avg_slippage_bps"),
        pl.col("slippage_bps").std().round(2).alias("std_slippage_bps"),
        pl.col("slippage_bps").count().alias("trade_count"),
    ])
    .sort("avg_slippage_bps")
)

In [ ]:
# Bar chart: average slippage by counterparty
cp_slippage = (
    slippage
    .group_by("counterparty")
    .agg(pl.col("slippage_bps").mean().round(2).alias("avg_slippage_bps"))
    .sort("avg_slippage_bps")
)

counterparties = cp_slippage["counterparty"].to_list()
avg_vals = cp_slippage["avg_slippage_bps"].to_list()
colors = ["green" if v <= 0 else "salmon" for v in avg_vals]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(counterparties, avg_vals, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Average Slippage (bps)")
ax.set_ylabel("Counterparty")
ax.set_title("Execution Slippage by Counterparty")
ax.axvline(x=0, color="gray", linewidth=0.5)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

## 6. RFQ Flow Analytics

Load the computed `rfq_analytics` features to analyze quantity trends and volatility patterns.

In [ ]:
rfq_feat_df = load_features("rfq_events", "rfq_analytics").collect()

print(f"Shape: {rfq_feat_df.shape}")
print(f"Columns: {rfq_feat_df.columns}")

# Show features for the first few events
print("\nRFQ analytics features:")
print(
    rfq_feat_df.select(
        ["event_id", "rfq_id", "counterparty", "requested_qty", "qty_sma_5", "qty_rolling_vol_3"]
    ).head(15)
)

# Quantity analytics by counterparty
print("\n=== Quantity Analytics by Counterparty ===")
print(
    rfq_feat_df
    .filter(pl.col("qty_sma_5").is_not_null())
    .group_by("counterparty")
    .agg([
        pl.col("requested_qty").mean().round(4).alias("avg_qty"),
        pl.col("qty_sma_5").mean().round(4).alias("avg_qty_sma"),
        pl.col("qty_rolling_vol_3").mean().round(4).alias("avg_qty_vol"),
    ])
    .sort("avg_qty", descending=True)
)

## 7. Instrument Breakdown

Analyze trade volume and notional value by instrument to understand flow composition.

In [ ]:
# Compute notional value (price * quantity) for each trade
instrument_stats = (
    trade_df
    .with_columns(
        (pl.col("price") * pl.col("quantity")).alias("notional")
    )
    .group_by("instrument")
    .agg([
        pl.col("trade_id").count().alias("trade_count"),
        pl.col("quantity").sum().round(4).alias("total_qty"),
        pl.col("notional").sum().round(2).alias("total_notional"),
        pl.col("price").mean().round(2).alias("avg_price"),
    ])
    .sort("total_notional", descending=True)
)

print("=== Trade Volume by Instrument ===")
print(instrument_stats)

# Side-by-side bar charts: trade count and total notional by instrument
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

instruments = instrument_stats["instrument"].to_list()
trade_counts = instrument_stats["trade_count"].to_list()
notionals = instrument_stats["total_notional"].to_list()
colors = [plt.cm.Set2(i) for i in range(len(instruments))]

axes[0].bar(instruments, trade_counts, color=colors, edgecolor="black")
axes[0].set_title("Trade Count by Instrument")
axes[0].set_ylabel("Number of Trades")

axes[1].bar(instruments, notionals, color=colors, edgecolor="black")
axes[1].set_title("Total Notional by Instrument")
axes[1].set_ylabel("Notional (USD)")

plt.tight_layout()
plt.show()

## Key Findings

This analysis demonstrates the cross-dataset workflow pattern in ADP:

1. **RFQ Lifecycle** — Each RFQ flows through requested -> quoted -> accepted/rejected, with timing data at each stage
2. **Counterparty Analysis** — Acceptance rates and response times vary significantly by counterparty
3. **Execution Quality** — Slippage between quoted and executed prices can be measured in basis points to evaluate market maker quality
4. **Flow Analytics** — Computed features (qty_sma, qty_rolling_vol) reveal quantity patterns per counterparty
5. **Instrument Mix** — BTCUSDT and ETHUSDT flows have different volumes and notional sizes

These workflows are typical for OTC desk analysts monitoring counterparty relationships and execution quality.